In [1]:
import pandas as pd
from pathlib import Path
from sqlalchemy import create_engine, text
from dotenv import load_dotenv
import os

load_dotenv("../.env")
db_password = os.getenv("DB_PASSWORD")

DB_USER = "postgres"
DB_HOST = "localhost"
DB_PORT = "5432"
DB_NAME = "climate_agri_intelligence"

engine = create_engine(f"postgresql+psycopg2://{DB_USER}:{db_password}@{DB_HOST}:{DB_PORT}/{DB_NAME}")

with engine.connect() as conn:
    result = conn.execute(text("SELECT version();"))
    print(result.fetchone())

('PostgreSQL 18.6 on x86_64-windows, compiled by msvc-19.44.35228, 64-bit',)


In [2]:
PROCESSED_DIR = Path("../data/processed")

countries_df = pd.read_csv(PROCESSED_DIR / "countries_clean.csv")
calendar_df = pd.read_csv(PROCESSED_DIR / "calendar_clean.csv")
climate_df = pd.read_csv(PROCESSED_DIR / "climate_clean.csv")
weather_df = pd.read_csv(PROCESSED_DIR / "weather_clean.csv")
population_df = pd.read_csv(PROCESSED_DIR / "population_clean.csv")
crop_yields_df = pd.read_csv(PROCESSED_DIR / "crop_yields_clean.csv")

for name, df in [("countries", countries_df), ("calendar", calendar_df), ("climate", climate_df),
                  ("weather", weather_df), ("population", population_df), ("crop_yields", crop_yields_df)]:
    print(f"{name}: {df.shape}")

countries: (177, 5)
calendar: (29, 4)
climate: (177, 6)
weather: (5133, 5)
population: (4901, 3)
crop_yields: (13745, 7)


In [4]:
bad_rows = countries_df[countries_df['iso3'] == '-99']
print("Rows with the invalid placeholder code '-99':")
print(bad_rows[['country_name', 'iso3']].to_string())

countries_df = countries_df[countries_df['iso3'] != '-99'].copy()
weather_df = weather_df[weather_df['iso3'] != '-99'].copy()
climate_df = climate_df[climate_df['iso3'] != '-99'].copy()

print(f"\nAfter removing: countries {countries_df.shape}, weather {weather_df.shape}, climate {climate_df.shape}")

countries_df.to_csv(PROCESSED_DIR / "countries_clean.csv", index=False)
weather_df.to_csv(PROCESSED_DIR / "weather_clean.csv", index=False)
climate_df.to_csv(PROCESSED_DIR / "climate_clean.csv", index=False)
print("Processed files overwritten with the fix.")

Rows with the invalid placeholder code '-99':
    country_name iso3
160    N. Cyprus  -99
167   Somaliland  -99
174       Kosovo  -99

After removing: countries (174, 5), weather (5046, 5), climate (174, 6)
Processed files overwritten with the fix.


In [5]:
countries_load = countries_df[['iso3', 'country_name', 'continent', 'lon', 'lat']]
climate_load = climate_df[['iso3', 'baseline_avg_temp_c', 'baseline_avg_precip_mm', 'baseline_start', 'baseline_end']]
weather_load = weather_df[['iso3', 'year', 'avg_temp_c', 'total_precip_mm']]

print("'-99' in population iso3:", (population_df['iso3'] == '-99').any())
print("'-99' in crop_yields iso3:", (crop_yields_df['iso3'] == '-99').any())

load_order = [
    ('countries', countries_load),
    ('calendar', calendar_load),
    ('climate', climate_load),
    ('weather', weather_load),
    ('population', population_load),
    ('crop_yields', crop_yields_load),
]

'-99' in population iso3: False
'-99' in crop_yields iso3: False


In [6]:
reverse_order = ['crop_yields', 'population', 'weather', 'climate', 'calendar', 'countries']

with engine.begin() as conn:
    for table_name in reverse_order:
        conn.execute(text(f"TRUNCATE TABLE {table_name} CASCADE;"))
print("All tables truncated — starting from a clean, empty state.\n")

for table_name, df in load_order:
    df.to_sql(table_name, engine, if_exists='append', index=False)
    print(f"Loaded {len(df)} rows into {table_name}")

All tables truncated — starting from a clean, empty state.

Loaded 174 rows into countries
Loaded 29 rows into calendar
Loaded 174 rows into climate
Loaded 5046 rows into weather
Loaded 4901 rows into population
Loaded 13745 rows into crop_yields


In [7]:
with engine.connect() as conn:
    for table_name, _ in load_order:
        result = conn.execute(text(f"SELECT COUNT(*) FROM {table_name};"))
        count = result.scalar()
        print(f"{table_name}: {count} rows in Postgres")

countries: 174 rows in Postgres
calendar: 29 rows in Postgres
climate: 174 rows in Postgres
weather: 5046 rows in Postgres
population: 4901 rows in Postgres
crop_yields: 13745 rows in Postgres
